## Health checks from `01_checks.sql`

### $\chi^2$ Test: Are the group sizes consistent with an equal three-way split? 

In [1]:
import pandas as pd
from scipy.stats import chisquare

df = pd.read_csv("../data/raw/hillstrom.csv")

counts = df["segment"].value_counts()

print(chisquare(counts))

Power_divergenceResult(statistic=np.float64(0.20253125000000002), pvalue=np.float64(0.9036929575604196))


$\chi^2$ = 0.20, p = 0.904. Since p is well above our 0.01 threshold, there is no evidence that the split is off.

### Standardized mean difference: Are the groups alike in every way before the emails are sent?

Write function to calculate standardized mean difference.

In [2]:
def smd(a, b):
    return (a.mean() - b.mean()) / ((a.var() + b.var()) / 2) ** 0.5

Loop through all the groups to see if they are similar.

In [3]:
df["is_phone"]        = (df["channel"] == "Phone").astype(int)
df["is_web"]          = (df["channel"] == "Web").astype(int)
df["is_multichannel"] = (df["channel"] == "Multichannel").astype(int)

df["is_urban"]        = (df["zip_code"] == "Urban").astype(int)
df["is_suburban"]     = (df["zip_code"] == "Surburban").astype(int)
df["is_rural"]        = (df["zip_code"] == "Rural").astype(int)

group_a = df[df["segment"] == "Mens E-Mail"]
group_b = df[df["segment"] == "Womens E-Mail"]
control = df[df["segment"] == "No E-Mail"]

rows = []
for col in ["history", "recency", "newbie", "mens", "womens", "is_phone", "is_web", "is_multichannel", "is_urban", "is_suburban", "is_rural"]:
    rows.append({
        "characteristic": col,
        "mens_vs_none": smd(group_a[col], control[col]),
        "womens_vs_none": smd(group_b[col], control[col])
    })

table = pd.DataFrame(rows).round(3)
table

,characteristic,mens_vs_none,womens_vs_none
0,history,0.008,0.007
1,recency,0.007,0.005
2,newbie,-0.001,0.003
3,mens,-0.005,-0.009
4,womens,0.008,0.005
5,is_phone,-0.008,0.009
6,is_web,0.011,-0.005
7,is_multichannel,-0.004,-0.005
8,is_urban,0.002,-0.002
9,is_suburban,-0.012,-0.001


All values are below the $|SMD| < 0.1$ threshold, the largest value being 0.014 for Men's Email vs None for rural zip codes.

Health checks passed: no missing data, group sizes match the planned split, and the groups were balanced before the emails.